# Handwritten Digit Recognition (MNIST) with CNN + GUI
### Mithun V Gowda

### Install Dependencies

In [ ]:
pip install tensorflow pillow numpy

### Imports and Debug Flag

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import tkinter as tk
from PIL import Image, ImageOps, ImageFilter, ImageDraw

DEBUG = True

def log(msg):
    if DEBUG:
        print(msg)

np.random.seed(42)
tf.random.set_seed(42)

###  Load MNIST Dataset

In [ ]:
log("=== Loading MNIST Dataset ===")
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

log(f"x_train shape: {x_train.shape}")
log(f"y_train shape: {y_train.shape}")
log(f"x_test shape : {x_test.shape}")
log(f"y_test shape : {y_test.shape}")

### Preprocessing Dataset

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test  = np.expand_dims(x_test, -1)

num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat  = keras.utils.to_categorical(y_test, num_classes)

log("=== After Preprocessing ===")
log(f"x_train range: {x_train.min():.3f} → {x_train.max():.3f}")
log(f"x_train shape: {x_train.shape}")

### Data Augmentation

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
datagen.fit(x_train)

log("Data augmentation enabled.")

### Build CNN Model

In [ ]:
def build_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3,3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2D(32, (3,3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3,3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2D(64, (3,3), padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(0.5),

        layers.Dense(10, activation="softmax")
    ])
    return model

model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Train Model

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("mnist_best.keras", save_best_only=True, monitor="val_accuracy"),
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)
]

history = model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=128),
    steps_per_epoch=len(x_train)//128,
    validation_data=(x_test, y_test_cat),
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

### Evaluate Model

In [ ]:
score = model.evaluate(x_test, y_test_cat, verbose=0)
log(f"Test Accuracy: {score[1]*100:.2f}%")
log(f"Test Loss    : {score[0]:.4f}")

### Load Trained Model

In [ ]:
model = keras.models.load_model("mnist_best.keras")
log("Model loaded successfully.")

### Preprocessing for GUI Input

In [10]:
def preprocess_canvas_image(pil_img):
    log("\n=== Preprocessing ===")
    log(f"Original size: {pil_img.size}")

    img = ImageOps.invert(pil_img.convert("L"))
    img = img.filter(ImageFilter.GaussianBlur(radius=1))

    bw = img.point(lambda x: 255 if x > 30 else 0, mode="1")
    bbox = bw.getbbox()
    if bbox is None:
        log("No digit detected.")
        return None

    img = img.crop(bbox)
    w, h = img.size
    side = max(w, h)

    square = Image.new("L", (side, side), 0)
    square.paste(img, ((side-w)//2, (side-h)//2))

    square = square.resize((20,20), Image.Resampling.LANCZOS)
    final = Image.new("L", (28,28), 0)
    final.paste(square, (4,4))

    return final

### Prediction Function

In [11]:
def predict_digit(pil_img):
    proc = preprocess_canvas_image(pil_img)
    if proc is None:
        return None, 0.0

    arr = np.array(proc).astype("float32") / 255.0
    arr = arr.reshape(1, 28, 28, 1)

    probs = model.predict(arr, verbose=0)[0]
    digit = int(np.argmax(probs))
    conf = float(np.max(probs))

    log(f"Prediction: {digit}, Confidence: {conf*100:.2f}%")
    return digit, conf

### GUI Application (Tkinter)

In [ ]:
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Digit Recognizer")
        self.geometry("520x360")

        self.canvas_size = 300
        self.canvas = tk.Canvas(self, width=300, height=300, bg="white")
        self.canvas.grid(row=0, column=0, padx=10, pady=10)

        right = tk.Frame(self)
        right.grid(row=0, column=1, sticky="n")

        self.label = tk.Label(right, text="Draw a digit", font=("Helvetica", 24))
        self.label.pack(pady=10)

        tk.Button(right, text="Recognise", command=self.classify).pack(fill="x")
        tk.Button(right, text="Clear", command=self.clear).pack(fill="x", pady=5)

        self.r = 5
        self.image = Image.new("L", (300,300), 255)
        self.draw = ImageDraw.Draw(self.image)

        self.canvas.bind("<B1-Motion>", self.paint)

    def paint(self, event):
        x, y = event.x, event.y
        self.canvas.create_oval(x-self.r, y-self.r, x+self.r, y+self.r, fill="black")
        self.draw.ellipse([x-self.r, y-self.r, x+self.r, y+self.r], fill=0)

    def clear(self):
        self.canvas.delete("all")
        self.label.config(text="Draw a digit")
        self.image = Image.new("L", (300,300), 255)
        self.draw = ImageDraw.Draw(self.image)

    def classify(self):
        digit, conf = predict_digit(self.image)
        if digit is None:
            self.label.config(text="Draw!")
        else:
            self.label.config(text=f"{digit} ({int(conf*100)}%)")

App().mainloop()